# Strategy 2. Template-based query

Hand-code query templates and only allow the LLM to (a) select the right template from a finite collection of templates using the natural language prompt and (b) populate the plug-in parameters in these query templates, also using information supplied in the prompt.

Single query is technically sufficient to produce correct answers for all questions, so we will use generate other query templates as decoys. 

In the simplest implementation queries will be hidden from LLM. LLM will be asked to a) pick query, and b) provide commands for query in json format. For a more complicated models this could be organized as a tool call.

**Main query example**:

```cypher
MATCH (gene:HumanGene)-[:IS_PART_OF]->(assoc)<-[:IS_PART_OF]-(disease:Disease)
WHERE gene.approvedSymbol = 'TARDBP' 
  AND (LOWER(disease.name) CONTAINS LOWER('amyotrophic lateral sclerosis') OR LOWER(disease.name) CONTAINS LOWER('ALS'))
  AND assoc.score IS NOT NULL
  AND (assoc:`AnimalModel.GeneToDiseaseAssociation` OR assoc:`Literature.GeneToDiseaseAssociation`)
  AND assoc.score > 0.3
RETURN 
    gene.approvedSymbol as Gene,
    disease.name as Disease,
    head([label IN labels(assoc) WHERE label CONTAINS '.GeneToDiseaseAssociation']) as EvidenceType,
    assoc.score as Score,
    assoc.literature as Literature
ORDER BY assoc.score DESC
```

Three sub-strategies are evaluated:
- 2a - with a single option (to understand if LLM picks parameters correctly), 
- 2b - with decoy queries and examples of correct query
- 2c - with decoy queries and without examples

In [59]:
import os
from dotenv import load_dotenv
from tqdm import tqdm
import json
import pandas as pd

# Load environment variables from the .env file
load_dotenv()

# Retrieve the variables from the environment
neo4j_uri = os.getenv("NEO4J_URI")
neo4j_username = os.getenv("NEO4J_USERNAME")
neo4j_password = os.getenv("NEO4J_PASSWORD")

# Check if any variable is missing
if not all([neo4j_uri, neo4j_username, neo4j_password]):
    raise EnvironmentError("One or more environment variables are missing: NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD")

print(f"Accessing OpenTargets at {neo4j_uri} as user {neo4j_username}")

Accessing OpenTargets at bolt+s://pistoia.neo4j.rbsapp.net:7687 as user neo4j


Wrappers for LLMs and KG

In [60]:
# Langchain wrappers for different models

import re
from langchain_openai import ChatOpenAI
from langchain_mistralai import ChatMistralAI
from langchain_anthropic import ChatAnthropic

chat_models = {}

def new_ChatModel(model):
    if re.search(r"^gpt", model):
        return ChatOpenAI(model = model)
    elif re.search(r"^o1", model):
        return ChatOpenAI(model = model, temperature = 1)
    elif re.search(r"^claude", model):
        return ChatAnthropic(model = model)
    elif re.search(r"mistral", model):
        return ChatMistralAI(model = model)
    else:
        raise ValueError(f"Unsupported model: {model}")

def ChatModel(model):
    if model in chat_models:
        return chat_models[model]
    else:
        chat_models[model] = new_ChatModel(model)
        return chat_models[model]

# models:
#   claude-sonnet-4-6
#   gpt-4o
#   o1-preview-2024-09-12
#   open-mistral-7b

# llm = ChatModel("gpt-4o")


In [61]:
# Json data extraction

def extract_json(message):
    text = message
    pattern = r"```json(.*?)```"
    matches = re.findall(pattern, text, re.DOTALL)
    try:
        return [json.loads(match.strip()) for match in matches]
    except Exception:
        raise ValueError(f"Failed to parse: {message}")

from py2neo import Graph

graph = Graph(
    neo4j_uri,
    auth=(neo4j_username, neo4j_password)
)



In [62]:
# Some queries take a very long time to run. This will deal with timeout

import threading

class TimeoutThread(threading.Thread):
    def __init__(self, func, *args, **kwargs):
        threading.Thread.__init__(self)
        self.func = func
        self.args = args
        self.kwargs = kwargs
        self.result = None
        self.error = None

    def run(self):
        try:
            self.result = self.func(*self.args, **self.kwargs)
        except Exception as e:
            self.error = e

def run_with_timeout(func, timeout, *args, **kwargs):
    thread = TimeoutThread(func, *args, **kwargs)
    thread.start()
    thread.join(timeout)

    if thread.is_alive():
        raise TimeoutError("Function execution timed out")
    elif thread.error:
        raise thread.error
    return thread.result

### Generating cypher query from query name and parameters in json

We'll hard-code cypher queries generation in this strategy. LLM should produce the output like this:

```json
{
    "query_name": "gene_disease_association",
    "query_params": {
        "gene_symbol" : "TARDBP",
        "disease_name" : ["amyotrophic lateral sclerosis", "als"],
        "evidence_type" : ["Literature", "AnimalModel"],
        "evidence_threshold" : 0.3
    }
}
```

Cypher queries will be generated using query_params.


In [63]:
# generating query from json schema

template_gene_disease_query = """
MATCH (gene:HumanGene)-[:IS_PART_OF]->(assoc)<-[:IS_PART_OF]-(disease:Disease)
WHERE gene.approvedSymbol = '{gene_symbol}' 
  AND {disease_names}
  AND assoc.score IS NOT NULL
  {evidence_type}
  {score_threshold}
RETURN 
    gene.approvedSymbol as Gene,
    disease.name as Disease,
    head([label IN labels(assoc) WHERE label CONTAINS '.GeneToDiseaseAssociation']) as EvidenceType,
    assoc.score as Score,
    assoc.literature as Literature
ORDER BY assoc.score DESC

"""



def generate_cypher_gene_disease_association(params):
    
    gene_symbol = params['gene_symbol']

    # disease:
    # (LOWER(disease.name) CONTAINS LOWER('amyotrophic lateral sclerosis') OR LOWER(disease.name) CONTAINS LOWER('ALS'))    
    diseases = params['disease_name']
    if len(diseases) == 0:
        raise ValueError(f"diseases list should not be empty in query: {params}")
    else:
        disease_names = "(" + " OR ".join([f"LOWER(disease.name) CONTAINS '{d.lower()}'" for d in diseases]) + ")"
    
    # evidence_type:
    #AND (assoc:`AnimalModel.GeneToDiseaseAssociation` OR assoc:`Literature.GeneToDiseaseAssociation`)
    evidence_type = ""
    if 'evidence_type' in params:
        if len(params['evidence_type']) > 0:
            evidence_type = "AND (" +  " OR ".join([f"assoc:`{t}.GeneToDiseaseAssociation`" for t in params['evidence_type']])  + ")"

    # score_threshold:
    #AND assoc.score > 0.3
    score_threshold = ""
    if 'score_threshold' in params:
        score_threshold = f"AND assoc.score > {params['score_threshold']}"
    
    return template_gene_disease_query.format(gene_symbol=gene_symbol, disease_names=disease_names, evidence_type=evidence_type, score_threshold=score_threshold)
    
        
def generate_cypher_mock_query(query_name, params):
    raise ValueError(f"Requested decoy query {query_name}")


def generate_cypher(query):

    query_name = query['query_name']
    query_params = query['query_params']

    if query_name == 'gene_disease_association':
        return generate_cypher_gene_disease_association(query_params)
    elif query_name == 'drug_target_association':
        return generate_cypher_mock_query(query_name, query_params)
    elif query_name == 'drug_disease_association':
        return generate_cypher_mock_query(query_name, query_params)
    
    else:
        raise ValueError(f"Incorrect query name {query_name}")
    
    



In [64]:
# convenience functions for data retrieval from graph

import time

def query_cypher_graph(graph, query):
    return graph.query(query)

def query_graph(llm_output):
    try:
        json_queries = extract_json(llm_output)
        cypher_query = [generate_cypher(json_query) for json_query in json_queries]
    except Exception as e:
        return [{
            "query":None,
            "success":False,
            "exception":str(e)
        }]

    cypher_results = []
    for query in cypher_query:
        try:
            start = time.time()
            result = run_with_timeout(query_cypher_graph, 20, graph, query)
            duration = time.time() - start
            cypher_results.append({
                "query":query,
                "success":True,
                "result": list(result),
                "time": duration
                })
        except Exception as e:
            cypher_results.append({
                "query":query,
                "success":False,
                "exception":str(e)
            })
    return cypher_results


def process_results(todo, llm_answers, cypher_results):
    results = []
    for t,llm,res in zip(todo, llm_answers, cypher_results):
        out = {
            "model" : t[0],
            "question" : t[1],
            "llm_answer" : llm,
            "cypher_output": res,
            "n_cypher_queries" : len(res)
        }
        if len(res) > 0:
            out.update({
                "query": res[0].get('query',''),
                "success": res[0]['success']
            })
            if res[0]['success']:
                out.update({
                    "results": res[0]['result'],
                    "time": res[0]['time'],
                    "count" : len(res[0]['result'])
                })
            else:
                out.update({
                    "error": res[0]['exception']
                })
        results.append(out)
    return results



In [65]:
# testing it:

def test_query():

    test_outputs = """
This is your output:

```json
{
    "query_name": "gene_disease_association",
    "query_params": {
        "gene_symbol" : "TARDBP",
        "disease_name" : ["amyotrophic lateral sclerosis", "als"],
        "evidence_type" : ["Literature", "AnimalModel"],
        "evidence_threshold" : 0.3
    }
}
```


```json
{
    "query_name": "gene_disease_association",
    "query_params": {
        "gene_symbol" : "BRAF",
        "disease_name" : ["melanoma"]
    }
}
```

"""
    res = query_graph(test_outputs)
    print(res)
    
#test_query()


## Questions

In [66]:
questions = [
    "What (or how strong, or is there any) is the evidence between TDP-43 and amyotrophic lateral sclerosis (ALS)",
    "What is the evidence linking TDP-43 to cancer in animal models?",
    "What (or is there) is the clinical evidence linking BRAF to Melanoma?"
]

## Running template-based query

We'll make 3 queries: 
1) with a single option (to understand if LLM picks parameters correctly), 
2) with many options and examples (to see how presence of other queries confuses LLM)
3) with many options, and without examples

The overall approach could be easily generalized to enable support of validated json schemas

In [67]:
from langchain_core.prompts import PromptTemplate


# Query building blocks:

gene_description = """\
field_name: gene_symbol
field_type: mandatory, string
field_value: official HGCN-approved symbol of human gene, e.g. "KRAS"\
"""

disease_description = """\
field_name: disease_name
field_type: mandatory, list of strings
field_value: a non-empty list of diseases that you want to retrieve an association with. for example, ["lung cancer", "colon cancer"]. The algorithm with match substrings, so if you are not sure about correct spelling, use all possible variants you can think of. Search is case-insensitive.\
"""

evidence_type_description = """\
field_name: evidence_type
field_type: optional, list of strings
field_value: specify if you want to restrict associations by evidence type. List all evidence types you want to include. For gene-disease associations allowed values are: AffectedPathway, AnimalModel, GeneticAssociation, KnownDrug, Literature, RnaExpression, SomaticMutation. If you don't want restriction, omit this field. Note that other fields may have other allowed evidence types.\
"""

evidence_threshold_description = """\
field_name: evidence_threshold
field_type: optional, double between 0 and 1
field_value: specify the threshold for evidence score. The lower score the worse the evidence. Provide minimal value for the evidence to be printed out. If you omit the value, all associations will be printed.\
"""

# Output examles:

query_example_gene_disease_association = """\
Input: Associations between KRAS and cancer, rna expression only, and evidence score >= 0.1
Output: 

```json
{
    "query_name": "gene_disease_association",
    "query_params": {
        "gene_symbol" : "KRAS",
        "disease_name" : ["cancer", "carcinoma"],
        "evidence_type" : ["RnaExpression"],
        "evidence_threshold" : 0.1
    }
}
```\
"""

query_example_drug_target_association = """\
Input: Clinically investigated or approved HDAC1 drugs.
Output: 

```json
{
    "query_name": "drug_target_association",
    "query_params": {
        "target_name" : "HDAC1",
        "evidence_type" : ["ApprovedDrug", "ClinicalStudy"]
    }
}
```\
"""

query_example_drug_indication_association = """\
Input: Approved indications of crizotinib.
Output: 

```json
{
    "query_name": "drug_disease_association",
    "query_params": {
        "drug_name" : "crizotinib",
        "evidence_type" : ["ApprovedDrug"]
    }
}
```\
"""

# Descriptions:

description_gene_disease_association = f"""\
**gene_disease_association**

This query returns associations between genes and diseases, and types of evidence

{gene_description}

{disease_description}

{evidence_type_description}

{evidence_threshold_description}\
"""

description_drug_target_association = f"""\
**drug_target_association**

This query returns associations between drugs and their targets

field_name: drug_name
field_type: mandatory, string
field_value: name of the drug. If not specified, query will return all drugs associated with a target

field_name: target_name
field_type: mandatory, string
field_value: official HGCN-approved symbol of human target, e.g. "KRAS". If not specified, query will return all targets associated with drugs.

field_name: evidence_type
field_type: optional, list of strings
field_value: specify if you want to restrict associations by evidence type. List all evidence types you want to include. Allowed values are: ClinicalStudy, AnimalModel, InVitroModel, ApprovedDrug. If you don't want restriction, omit this field.  Note that other fields may have other allowed evidence types.

{evidence_threshold_description}\
"""

description_drug_disease_association = f"""\
**drug_disease_association**

This query returns associations between drugs and diseases they intend to treat

field_name: drug_name
field_type: mandatory, string
field_value: name of the drug

{disease_description}

field_name: evidence_type
field_type: optional, list of strings
field_value: specify if you want to restrict associations by evidence type. List all evidence types you want to include. Allowed values are: ClinicalStudy, AnimalModel, InVitroModel, ApprovedDrug. If you don't want restriction, omit this field.  Note that other fields may have other allowed evidence types.\
"""

description_single_query = f"""\
{description_gene_disease_association}\
"""

description_multiple_queries = f"""\
{description_gene_disease_association}

{description_drug_target_association}

{description_drug_disease_association}\
"""

example_single_query = f"""\
Here is an example of correct json:

{query_example_gene_disease_association}
--------------------------------------------    
"""

example_multiple_query = f"""\
Here are few examples of correct json:

{query_example_gene_disease_association}

{query_example_drug_target_association}

{query_example_drug_indication_association}
--------------------------------------------    
"""



system_prompt_generic = """
You are a biological data scientist with vast experience in building and extracting information from knowledge graph. 

Your current project is to support scientists who want to answer scientific questions about genes, diseases and drugs. You have a database with biological data that accepts queries in a json format and returns the results.

The json should have 2 mandatory fields: query_name and query_params:

field_name: query_name
field_value: name of the query. See allowed values and descriptions below

field_name: query_params
field_value: a dictionary with parameters specific for each query (see below)

Here are parameters for each individual query:

--------------------------------------------
{query_descriptions}
--------------------------------------------
{examples}

Scientists will give you question, and you will generate a json query that will help to answer the question
"""

system_prompt_single_query = system_prompt_generic.format(query_descriptions = description_single_query, examples = example_single_query)
system_prompt_multiple_queries = system_prompt_generic.format(query_descriptions = description_multiple_queries, examples = example_multiple_query)
system_prompt_multiple_zeroshot = system_prompt_generic.format(query_descriptions = description_multiple_queries, examples = '')

template_query = """
{question}
"""

user_template = PromptTemplate.from_template(template_query)


# Option 2a - single query

There is only one query in the list of queries. With this approach we will test how good LLMs are at picking correct parameters for queries to run

In [68]:
from langchain.schema import AIMessage, HumanMessage, SystemMessage

models = ["gpt-4o", "claude-sonnet-4-6", "open-mistral-7b", "o1-preview-2024-09-12"]
niter = 10

todo = [(m, q) for q in questions for m in models for _ in range(niter)]

llm_answers = []
for llm_model, question in tqdm(todo, desc="Prompting LLM"):
    try:
        llm = ChatModel(model = llm_model)
        user_prompt = user_template.invoke({"question": question})

        # o1-preview does not support system messages, so joining them together:
        if llm_model == 'o1-preview-2024-09-12':
            messages = f"{system_prompt_single_query}\n------------------------------------------\nUser question:\n{user_prompt.text}\n"
            result = llm.invoke(messages)
        else:
            messages = [
                SystemMessage(content=system_prompt_single_query),
                HumanMessage(content=user_prompt.text)
            ]
            result = llm.invoke(messages)

        llm_answers.append(result.content)

    except Exception as e:
       print(e)
       llm_answers.append(None)

Prompting LLM:  22%|██▏       | 26/120 [01:39<03:15,  2.08s/it]

Error response 429 while fetching https://api.mistral.ai/v1/chat/completions: {"message":"Requests rate limit exceeded"}


Prompting LLM:  25%|██▌       | 30/120 [01:45<01:51,  1.24s/it]

Error response 429 while fetching https://api.mistral.ai/v1/chat/completions: {"message":"Requests rate limit exceeded"}
Error response 429 while fetching https://api.mistral.ai/v1/chat/completions: {"message":"Requests rate limit exceeded"}


Prompting LLM:  54%|█████▍    | 65/120 [05:03<01:34,  1.71s/it]

...

Prompting LLM:  88%|████████▊ | 106/120 [08:38<00:19,  1.39s/it]

Error response 429 while fetching https://api.mistral.ai/v1/chat/completions: {"message":"Requests rate limit exceeded"}
Error response 429 while fetching https://api.mistral.ai/v1/chat/completions: {"message":"Requests rate limit exceeded"}


Prompting LLM:  90%|█████████ | 108/120 [08:40<00:13,  1.11s/it]

Error response 429 while fetching https://api.mistral.ai/v1/chat/completions: {"message":"Requests rate limit exceeded"}


Prompting LLM: 100%|██████████| 120/120 [11:00<00:00,  5.50s/it]


In [71]:
import time
for i, (llm_model, question) in enumerate(todo):
    if not llm_answers[i]:
        try:
            llm = ChatModel(model = llm_model)
            user_prompt = user_template.invoke({"question": question})

            # o1-preview does not support system messages, so joining them together:
            if llm_model == 'o1-preview-2024-09-12':
                messages = f"{system_prompt_single_query}\n------------------------------------------\nUser question:\n{user_prompt.text}\n"
                result = llm.invoke(messages)
            else:
                messages = [
                    SystemMessage(content=system_prompt_single_query),
                    HumanMessage(content=user_prompt.text)
                ]
                result = llm.invoke(messages)

            llm_answers[i] = result.content

        except Exception as e:
            time.sleep(2)
            print(e)
            llm_answers[i] = None


In [72]:
cypher_results = []
for answer in tqdm(llm_answers, desc="Querying graph"):
    cypher_results.append(query_graph(answer))

Querying graph: 100%|██████████| 120/120 [00:26<00:00,  4.45it/s]


In [73]:
results = process_results(todo, llm_answers, cypher_results)

results_df = pd.DataFrame(results)
results_df.to_excel("02a-evaluations.xlsx", index=False)
with open("02a-evaluations.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=4)
results_df

,model,question,llm_answer,cypher_output,n_cypher_queries,query,success,results,time,count,error
0,gpt-4o,"What (or how strong, or is there any) is the e...","To address this question, we need to query the...",[{'query': ' MATCH (gene:HumanGene)-[:IS_PART_...,1,\nMATCH (gene:HumanGene)-[:IS_PART_OF]->(assoc...,True,[],0.138555,0.0,NaN
1,gpt-4o,"What (or how strong, or is there any) is the e...","To address this question, we need to query the...",[{'query': ' MATCH (gene:HumanGene)-[:IS_PART_...,1,\nMATCH (gene:HumanGene)-[:IS_PART_OF]->(assoc...,True,[],0.142533,0.0,NaN
2,gpt-4o,"What (or how strong, or is there any) is the e...",To address the question regarding the evidence...,[{'query': ' MATCH (gene:HumanGene)-[:IS_PART_...,1,\nMATCH (gene:HumanGene)-[:IS_PART_OF]->(assoc...,True,[],0.135524,0.0,NaN
3,gpt-4o,"What (or how strong, or is there any) is the e...",To address your question about the evidence be...,[{'query': ' MATCH (gene:HumanGene)-[:IS_PART_...,1,\nMATCH (gene:HumanGene)-[:IS_PART_OF]->(assoc...,True,[],0.142088,0.0,NaN
4,gpt-4o,"What (or how strong, or is there any) is the e...",To address the question regarding the evidence...,[{'query': ' MATCH (gene:HumanGene)-[:IS_PART_...,1,\nMATCH (gene:HumanGene)-[:IS_PART_OF]->(assoc...,True,[],0.139563,0.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...
115,o1-preview-2024-09-12,What (or is there) is the clinical evidence li...,To find the clinical evidence linking **BRAF**...,[{'query': ' MATCH (gene:HumanGene)-[:IS_PART_...,1,\nMATCH (gene:HumanGene)-[:IS_PART_OF]->(assoc...,True,"[(BRAF, melanoma, KnownDrug.GeneToDiseaseAssoc...",0.174045,376.0,NaN
116,o1-preview-2024-09-12,What (or is there) is the clinical evidence li...,"```json\n{\n ""query_name"": ""gene_disease_as...",[{'query': ' MATCH (gene:HumanGene)-[:IS_PART_...,1,\nMATCH (gene:HumanGene)-[:IS_PART_OF]->(assoc...,True,"[(BRAF, melanoma, KnownDrug.GeneToDiseaseAssoc...",0.172045,376.0,NaN
117,o1-preview-2024-09-12,What (or is there) is the clinical evidence li...,"```json\n{\n ""query_name"": ""gene_disease_as...",[{'query': ' MATCH (gene:HumanGene)-[:IS_PART_...,1,\nMATCH (gene:HumanGene)-[:IS_PART_OF]->(assoc...,True,"[(BRAF, melanoma, KnownDrug.GeneToDiseaseAssoc...",0.330573,5806.0,NaN
118,o1-preview-2024-09-12,What (or is there) is the clinical evidence li...,"```json\n{\n ""query_name"": ""gene_disease_as...",[{'query': ' MATCH (gene:HumanGene)-[:IS_PART_...,1,\nMATCH (gene:HumanGene)-[:IS_PART_OF]->(assoc...,True,"[(BRAF, melanoma, KnownDrug.GeneToDiseaseAssoc...",0.165036,376.0,NaN


# Option 2b - multiple queries

One correct query and 2 decoy queries + json examples

In [74]:
from langchain.schema import AIMessage, HumanMessage, SystemMessage

models = ["gpt-4o", "claude-sonnet-4-6", "open-mistral-7b", "o1-preview-2024-09-12"]
niter = 10

todo = [(m, q) for q in questions for m in models for _ in range(niter)]


def run_llm_multiple(llm_model, question):
    try:
        llm = ChatModel(model = llm_model)
        user_prompt = user_template.invoke({"question": question})

        # o1-preview does not support system messages, so joining them together:
        if llm_model == 'o1-preview-2024-09-12':
            messages = f"{system_prompt_multiple_queries}\n------------------------------------------\nUser question:\n{user_prompt.text}\n"
            result = llm.invoke(messages)
        else:
            messages = [
                SystemMessage(content=system_prompt_multiple_queries),
                HumanMessage(content=user_prompt.text)
            ]
            result = llm.invoke(messages)
        return result.content
    except Exception as e:
        print(e)
        return None


llm_answers = []
for llm_model, question in tqdm(todo, desc="Prompting LLM"):
    llm_answers.append(run_llm_multiple(llm_model, question))


Prompting LLM:  22%|██▎       | 27/120 [01:31<02:55,  1.89s/it]

Error response 429 while fetching https://api.mistral.ai/v1/chat/completions: {"message":"Requests rate limit exceeded"}


Prompting LLM:  52%|█████▎    | 63/120 [04:47<02:32,  2.67s/it]

Error response 429 while fetching https://api.mistral.ai/v1/chat/completions: {"message":"Requests rate limit exceeded"}


Prompting LLM:  54%|█████▍    | 65/120 [04:49<01:29,  1.63s/it]

...

Prompting LLM:  90%|█████████ | 108/120 [08:14<00:10,  1.20it/s]

Error response 429 while fetching https://api.mistral.ai/v1/chat/completions: {"message":"Requests rate limit exceeded"}


Prompting LLM:  92%|█████████▏| 110/120 [08:16<00:07,  1.33it/s]

Error response 429 while fetching https://api.mistral.ai/v1/chat/completions: {"message":"Requests rate limit exceeded"}


Prompting LLM: 100%|██████████| 120/120 [10:40<00:00,  5.34s/it]


In [75]:
import time
for i, (llm_model, question) in enumerate(todo):
    if not llm_answers[i]:
        llm_answers[i] = run_llm_multiple(llm_model, question)
        time.sleep(2)

In [76]:
cypher_results = []
for answer in tqdm(llm_answers, desc="Querying graph"):
    cypher_results.append(query_graph(answer))

Querying graph: 100%|██████████| 120/120 [00:20<00:00,  5.75it/s]


In [78]:
results = process_results(todo, llm_answers, cypher_results)

results_df = pd.DataFrame(results)
results_df.to_excel("02b-evaluations.xlsx", index=False)
with open("02b-evaluations.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=4)
results_df

,model,question,llm_answer,cypher_output,n_cypher_queries,query,success,results,time,count,error
0,gpt-4o,"What (or how strong, or is there any) is the e...",To address the question about the evidence bet...,[{'query': ' MATCH (gene:HumanGene)-[:IS_PART_...,1,\nMATCH (gene:HumanGene)-[:IS_PART_OF]->(assoc...,True,"[(TARDBP, amyotrophic lateral sclerosis, Genet...",0.426623,1444.0,NaN
1,gpt-4o,"What (or how strong, or is there any) is the e...",To address the question regarding the evidence...,[{'query': ' MATCH (gene:HumanGene)-[:IS_PART_...,1,\nMATCH (gene:HumanGene)-[:IS_PART_OF]->(assoc...,True,"[(TARDBP, amyotrophic lateral sclerosis, Genet...",0.194048,1444.0,NaN
2,gpt-4o,"What (or how strong, or is there any) is the e...","```json\n{\n ""query_name"": ""gene_disease_as...",[{'query': ' MATCH (gene:HumanGene)-[:IS_PART_...,1,\nMATCH (gene:HumanGene)-[:IS_PART_OF]->(assoc...,True,[],0.136528,0.0,NaN
3,gpt-4o,"What (or how strong, or is there any) is the e...","```json\n{\n ""query_name"": ""gene_disease_as...",[{'query': ' MATCH (gene:HumanGene)-[:IS_PART_...,1,\nMATCH (gene:HumanGene)-[:IS_PART_OF]->(assoc...,True,[],0.142534,0.0,NaN
4,gpt-4o,"What (or how strong, or is there any) is the e...",To answer the question about the evidence betw...,"[{'query': None, 'success': False, 'exception'...",1,None,False,NaN,NaN,NaN,Failed to parse: To answer the question about ...
...,...,...,...,...,...,...,...,...,...,...,...
115,o1-preview-2024-09-12,What (or is there) is the clinical evidence li...,"```json\n{\n ""query_name"": ""gene_disease_as...",[{'query': ' MATCH (gene:HumanGene)-[:IS_PART_...,1,\nMATCH (gene:HumanGene)-[:IS_PART_OF]->(assoc...,True,"[(BRAF, melanoma, KnownDrug.GeneToDiseaseAssoc...",0.189047,376.0,NaN
116,o1-preview-2024-09-12,What (or is there) is the clinical evidence li...,"```json\n{\n ""query_name"": ""gene_disease_as...",[{'query': ' MATCH (gene:HumanGene)-[:IS_PART_...,1,\nMATCH (gene:HumanGene)-[:IS_PART_OF]->(assoc...,True,"[(BRAF, melanoma, KnownDrug.GeneToDiseaseAssoc...",0.183555,376.0,NaN
117,o1-preview-2024-09-12,What (or is there) is the clinical evidence li...,"```json\n{\n ""query_name"": ""gene_disease_as...",[{'query': ' MATCH (gene:HumanGene)-[:IS_PART_...,1,\nMATCH (gene:HumanGene)-[:IS_PART_OF]->(assoc...,True,"[(BRAF, melanoma, KnownDrug.GeneToDiseaseAssoc...",0.160562,210.0,NaN
118,o1-preview-2024-09-12,What (or is there) is the clinical evidence li...,"```json\n{\n ""query_name"": ""gene_disease_as...",[{'query': ' MATCH (gene:HumanGene)-[:IS_PART_...,1,\nMATCH (gene:HumanGene)-[:IS_PART_OF]->(assoc...,True,"[(BRAF, melanoma, KnownDrug.GeneToDiseaseAssoc...",0.189069,376.0,NaN


# Option 2c - multiple queries, zero-shot

One correct query and 2 decoy queries, no json examples

In [79]:
from langchain.schema import AIMessage, HumanMessage, SystemMessage

models = ["gpt-4o", "claude-sonnet-4-6", "open-mistral-7b", "o1-preview-2024-09-12"]
niter = 10

todo = [(m, q) for q in questions for m in models for _ in range(niter)]


def run_llm_multiple_zeroshot(llm_model, question):
    try:
        llm = ChatModel(model = llm_model)
        user_prompt = user_template.invoke({"question": question})

        # o1-preview does not support system messages, so joining them together:
        if llm_model == 'o1-preview-2024-09-12':
            messages = f"{system_prompt_multiple_zeroshot}\n------------------------------------------\nUser question:\n{user_prompt.text}\n"
            result = llm.invoke(messages)
        else:
            messages = [
                SystemMessage(content=system_prompt_multiple_zeroshot),
                HumanMessage(content=user_prompt.text)
            ]
            result = llm.invoke(messages)
        return result.content
    except Exception as e:
        print(e)
        return None


llm_answers = []
for llm_model, question in tqdm(todo, desc="Prompting LLM"):
    llm_answers.append(run_llm_multiple_zeroshot(llm_model, question))


Prompting LLM:  54%|█████▍    | 65/120 [05:23<01:35,  1.73s/it]

Error response 429 while fetching https://api.mistral.ai/v1/chat/completions: {"message":"Requests rate limit exceeded"}
Error response 429 while fetching https://api.mistral.ai/v1/chat/completions: {"message":"Requests rate limit exceeded"}


Prompting LLM:  57%|█████▋    | 68/120 [05:25<00:47,  1.09it/s]

Error response 429 while fetching https://api.mistral.ai/v1/chat/completions: {"message":"Requests rate limit exceeded"}
Error response 429 while fetching https://api.mistral.ai/v1/chat/completions: {"message":"Requests rate limit exceeded"}


Prompting LLM:  58%|█████▊    | 70/120 [05:27<00:41,  1.22it/s]

...

Prompting LLM:  90%|█████████ | 108/120 [09:12<00:14,  1.21s/it]

Error response 429 while fetching https://api.mistral.ai/v1/chat/completions: {"message":"Requests rate limit exceeded"}
Error response 429 while fetching https://api.mistral.ai/v1/chat/completions: {"message":"Requests rate limit exceeded"}


Prompting LLM:  92%|█████████▏| 110/120 [09:12<00:06,  1.43it/s]

Error response 429 while fetching https://api.mistral.ai/v1/chat/completions: {"message":"Requests rate limit exceeded"}


Prompting LLM: 100%|██████████| 120/120 [11:46<00:00,  5.89s/it]


In [81]:
import time
for i, (llm_model, question) in enumerate(todo):
    if not llm_answers[i]:
        llm_answers[i] = run_llm_multiple_zeroshot(llm_model, question)
        time.sleep(2)

In [82]:
cypher_results = []
for answer in tqdm(llm_answers, desc="Querying graph"):
    cypher_results.append(query_graph(answer))

Querying graph: 100%|██████████| 120/120 [00:20<00:00,  5.75it/s]


In [83]:
results = process_results(todo, llm_answers, cypher_results)

results_df = pd.DataFrame(results)
results_df.to_excel("02c-evaluations.xlsx", index=False)
with open("02c-evaluations.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=4)
results_df

,model,question,llm_answer,cypher_output,n_cypher_queries,query,success,results,time,count,error
0,gpt-4o,"What (or how strong, or is there any) is the e...",To answer the question regarding the evidence ...,[{'query': ' MATCH (gene:HumanGene)-[:IS_PART_...,1,\nMATCH (gene:HumanGene)-[:IS_PART_OF]->(assoc...,True,"[(TARDBP, amyotrophic lateral sclerosis, Genet...",1.763236,1444.0,NaN
1,gpt-4o,"What (or how strong, or is there any) is the e...",To address the question regarding the evidence...,[{'query': ' MATCH (gene:HumanGene)-[:IS_PART_...,1,\nMATCH (gene:HumanGene)-[:IS_PART_OF]->(assoc...,True,"[(TARDBP, amyotrophic lateral sclerosis, Genet...",0.248111,1439.0,NaN
2,gpt-4o,"What (or how strong, or is there any) is the e...","To answer this question, we will use the `gene...",[{'query': ' MATCH (gene:HumanGene)-[:IS_PART_...,1,\nMATCH (gene:HumanGene)-[:IS_PART_OF]->(assoc...,True,[],0.135512,0.0,NaN
3,gpt-4o,"What (or how strong, or is there any) is the e...","To address this question, we need to perform a...",[{'query': ' MATCH (gene:HumanGene)-[:IS_PART_...,1,\nMATCH (gene:HumanGene)-[:IS_PART_OF]->(assoc...,True,[],0.135335,0.0,NaN
4,gpt-4o,"What (or how strong, or is there any) is the e...",To address the question regarding the evidence...,[{'query': ' MATCH (gene:HumanGene)-[:IS_PART_...,1,\nMATCH (gene:HumanGene)-[:IS_PART_OF]->(assoc...,True,"[(TARDBP, amyotrophic lateral sclerosis, Genet...",0.185041,1444.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...
115,o1-preview-2024-09-12,What (or is there) is the clinical evidence li...,"```json\n{\n ""query_name"": ""gene_disease_asso...",[{'query': ' MATCH (gene:HumanGene)-[:IS_PART_...,1,\nMATCH (gene:HumanGene)-[:IS_PART_OF]->(assoc...,True,"[(BRAF, nodular melanoma, SomaticMutation.Gene...",0.393091,5596.0,NaN
116,o1-preview-2024-09-12,What (or is there) is the clinical evidence li...,To determine if there is clinical evidence lin...,[{'query': ' MATCH (gene:HumanGene)-[:IS_PART_...,3,\nMATCH (gene:HumanGene)-[:IS_PART_OF]->(assoc...,True,"[(BRAF, melanoma, KnownDrug.GeneToDiseaseAssoc...",0.341806,5712.0,NaN
117,o1-preview-2024-09-12,What (or is there) is the clinical evidence li...,"```json\n{\n ""query_name"": ""gene_disease_asso...",[{'query': ' MATCH (gene:HumanGene)-[:IS_PART_...,1,\nMATCH (gene:HumanGene)-[:IS_PART_OF]->(assoc...,True,"[(BRAF, melanoma, KnownDrug.GeneToDiseaseAssoc...",0.359465,5806.0,NaN
118,o1-preview-2024-09-12,What (or is there) is the clinical evidence li...,"```json\n{\n ""query_name"": ""gene_disease_asso...",[{'query': ' MATCH (gene:HumanGene)-[:IS_PART_...,1,\nMATCH (gene:HumanGene)-[:IS_PART_OF]->(assoc...,True,"[(BRAF, superficial spreading melanoma, Somati...",0.255132,378.0,NaN
